# L09 – Lösningar
Kräver `dlkit/train.py` i L09-versionen.

In [ ]:
import sys
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders
from dlkit.train import fit, evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()
CKPT_DIR = Path("../checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))

train_loader, val_loader, _ = fashion_mnist_loaders()
X, y = next(iter(train_loader))
X_small, y_small = X[:32], y[:32]

### Övning 1

In [ ]:
set_seed(42)
model = make_mlp().to(device)
fit(model, train_loader, val_loader, loss_fn, torch.optim.Adam(model.parameters(), lr=1e-3),
    device, epochs=3, verbose=False)
acc_original = evaluate(model, val_loader, loss_fn, device)[1]

torch.save(model.state_dict(), CKPT_DIR / "ovning1.pt")
loaded = make_mlp().to(device)
loaded.load_state_dict(torch.load(CKPT_DIR / "ovning1.pt", map_location=device, weights_only=True))
acc_loaded = evaluate(loaded, val_loader, loss_fn, device)[1]   # evaluate anropar .eval()

assert acc_original == acc_loaded and acc_original > 0.8
print("Rätt!", acc_original)

### Övning 2
De tre buggarna:
1. **ReLU efter det sista lagret** klipper bort negativa logits. Ta bort den.
2. **Etiketterna blandas för sig**, så att `yb_shuffled` inte längre hör ihop med `Xb`. Använd `yb`.
3. **`opt.zero_grad()` saknas**, så gradienterna ackumuleras.

In [ ]:
set_seed(42)
fixed_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10),                     # 1. ingen ReLU på logits
).to(device)
opt = torch.optim.Adam(fixed_model.parameters(), lr=1e-3)
Xb, yb = X_small.to(device), y_small.to(device)   # 2. etiketterna i rätt ordning

for step in range(200):
    opt.zero_grad()                         # 3. nollställ gradienterna
    loss = loss_fn(fixed_model(Xb), yb)
    loss.backward()
    opt.step()
final_loss = loss.item()
assert final_loss < 0.05, final_loss
print("Rätt! slut-loss:", round(final_loss, 5))

### Övning 3

In [ ]:
set_seed(42)
model = make_mlp().to(device)
history = fit(model, train_loader, val_loader, loss_fn, torch.optim.Adam(model.parameters(), lr=1e-3),
              device, epochs=4, checkpoint_path=CKPT_DIR / "ovning3.pt")
ckpt = torch.load(CKPT_DIR / "ovning3.pt", map_location=device, weights_only=True)

best_idx = int(np.argmin(history["val_loss"]))
assert ckpt["epoch"] == best_idx + 1
assert abs(ckpt["val_loss"] - history["val_loss"][best_idx]) < 1e-9
print("Rätt! bästa epoken:", ckpt["epoch"])